
- Ficher source : [tco-billettique-frequentation-detaillee-td](https://data.explore.star.fr/explore/dataset/tco-billettique-frequentation-detaillee-td/table/?sort=datedebut)
- Fichier de sorti : int_star_id.csv
- Date de création : 13/11/2025
- Dernière modification : 17/11/2025

- Version(s) : 
    - 1 - 13/11/2025 : Nettoyage du fichier et création du csv "star_id.csv"
    - 2 - 17/11/2025 : Mise à jour des types des colonnes, suppression de la colonne "id_ligne_id_arret" et renommage du fichier csv "int_star_id"

In [1]:
#Librairie(s) utilisée(s)
import pandas as pd
import os

In [2]:
#Création d'une variable pour renseigner le chemin du répertoire stockant les fichiers
chemin = r"D:\Espace_de_travail_Jedha\02_dafs-ft-14\Projet\Bloc_6\1-Sources\1_Data_Bronze\FREQ-DETAIL"

#Création d'une boucle pour vérifier l'accès aux fichiers
fichiers = [f for f in os.listdir(chemin) 
            if os.path.isfile(os.path.join(chemin, f))]

#Tri de la liste pour ordonner les fichiers récupérés
fichiers = sorted(fichiers, reverse=True)

#Commande pour ne stocker que les X derniers fichiers en fonction de la date : LastYear = fichiers[-12:]

In [3]:
fichiers

['CLO.FREQ.EXTRACT.FREQ-DETAIL_202508.csv',
 'CLO.FREQ.EXTRACT.FREQ-DETAIL_202507.csv',
 'CLO.FREQ.EXTRACT.FREQ-DETAIL_202506.csv',
 'CLO.FREQ.EXTRACT.FREQ-DETAIL_202505.csv',
 'CLO.FREQ.EXTRACT.FREQ-DETAIL_202504.csv',
 'CLO.FREQ.EXTRACT.FREQ-DETAIL_202503.csv',
 'CLO.FREQ.EXTRACT.FREQ-DETAIL_202502.csv',
 'CLO.FREQ.EXTRACT.FREQ-DETAIL_202501.csv',
 'CLO.FREQ.EXTRACT.FREQ-DETAIL_202412.csv',
 'CLO.FREQ.EXTRACT.FREQ-DETAIL_202411.csv',
 'CLO.FREQ.EXTRACT.FREQ-DETAIL_202410.csv',
 'CLO.FREQ.EXTRACT.FREQ-DETAIL_202409.csv']

In [4]:
#Création d'une liste pour concaténer les données de chaque fichiers
combinaisons= []

#Création d'une boucle for pour filtrer uniquement les données souhaités puis de les mettre dans la liste
for fichier in fichiers:
    df= pd.read_csv(chemin + "\\" +  fichier, sep=";")

    #Création d'un filtre pour exclure Rennes des résultats et les valeurs vides de la colonne 'identifiantLigne'
    mask= (df['NomCommune'] != 'Rennes') & (df['identifiantLigne'].notna())
    df= df.loc[mask,['Timeo', 'identifiantLigne']]
    
    #Renommage des colonnes conservées
    df.rename(columns={'Timeo' : 'id_arret', 'identifiantLigne' : 'id_ligne' }, inplace= True)

    #Création d'une nouvelle colonne pour concaténer les deux afin d'en créer un id unique
    df['id_ligne_id_arret']= df['id_ligne'].astype(str) + '_' + df['id_arret'].astype(str)

    #Suppression des doublons
    df_ids_unique= df.drop_duplicates()

    #Ajout des données dans la liste 'combinaisons'
    combinaisons.append(df_ids_unique)

print(combinaisons[:5])

[        id_arret  id_ligne id_ligne_id_arret
16          2281         7            7_2281
17          2290         6            6_2290
18          2801         4            4_2801
19          2809         1            1_2809
20          3014        11           11_3014
...          ...       ...               ...
716659      4255        53           53_4255
736665      4705        72           72_4705
736992      3257        61           61_3257
738502      3216        61           61_3216
738503      3249        61           61_3249

[1239 rows x 3 columns],         id_arret  id_ligne id_ligne_id_arret
19        2237.0         6          6_2237.0
20        2281.0         7          7_2281.0
42        5050.0      1002       1002_5050.0
43        5051.0      1002       1002_5051.0
53        5063.0      1002       1002_5063.0
...          ...       ...               ...
687008    2252.0        37         37_2252.0
709401    2254.0        37         37_2254.0
756934    3809.0        74  

In [5]:
#Création du dataframe en combinant toutes les listes entre elles
df_star_id = pd.concat(combinaisons, ignore_index= True)
print(df_star_id.shape)
df_star_id.head()

(19059, 3)


,id_arret,id_ligne,id_ligne_id_arret
0,2281.0,7,7_2281
1,2290.0,6,6_2290
2,2801.0,4,4_2801
3,2809.0,1,1_2809
4,3014.0,11,11_3014


In [19]:
#Suppression de la colonne 'id_ligne_id_arret'
keep_col = ['id_arret', 'id_ligne']
df_star_id_unique = df_star_id[keep_col]

#Suppression des doublons de toutes les tables
df_star_id_unique= df_star_id_unique.drop_duplicates()

#Filtre sur les valeurs vides de la colonne 'id_arret'
mask = (df_star_id_unique['id_arret'].notna()) & (df_star_id_unique['id_arret'] != "")
df_star_id_unique = df_star_id_unique[mask]

In [22]:
# Changement de types
df_star_id_unique = df_star_id_unique.astype({
    'id_arret': 'int',
    'id_ligne': 'int'
})
df_star_id_unique.dtypes

id_arret    int64
id_ligne    int64
dtype: object

In [23]:
# Exporter le dataset nettoyé en csv
df_star_id_unique.to_csv("int_star_id.csv", sep=";", index=False)